# Graficos descritivos do catalogo de datasets

Notebook para gerar graficos simples para slides a partir de `catalogo_datasets.csv`.

Regra importante: a separacao entre binario e multiclasse usa somente a coluna `numero_classes`. Datasets com `numero_classes == 1` nao entram nos graficos de tipo, numero de classes, classe minoritaria ou classe majoritaria, porque nao sao binarios nem multiclasse.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

START_DIR = Path.cwd()
for candidate in (START_DIR, *START_DIR.parents):
    if (candidate / "descricao" / "catalogo_datasets.csv").exists():
        PROJECT_ROOT = candidate
        ROOT = candidate / "descricao"
        break
    if (candidate / "catalogo_datasets.csv").exists():
        ROOT = candidate
        PROJECT_ROOT = candidate.parent if candidate.name == "descricao" else candidate
        break
else:
    raise FileNotFoundError("catalogo_datasets.csv nao encontrado")

CATALOGO = ROOT / "catalogo_datasets.csv"
FIG_DIR = ROOT / "figuras_catalogo"
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CATALOGO)
df = df.copy()
df["numero_classes"] = pd.to_numeric(df["numero_classes"], errors="coerce")
df["numero_linhas"] = pd.to_numeric(df["numero_linhas"], errors="coerce")
df["features"] = pd.to_numeric(df["features"], errors="coerce")
df["classe_minoritaria_qtd"] = pd.to_numeric(df["classe_minoritaria_qtd"], errors="coerce")
df["classe_majoritaria_qtd"] = pd.to_numeric(df["classe_majoritaria_qtd"], errors="coerce")
df["percentual_ausentes"] = pd.to_numeric(df["percentual_ausentes"], errors="coerce").fillna(0)

df_classes = df[df["numero_classes"].between(2, 10)].copy()
df_classes["tipo_dataset"] = np.where(df_classes["numero_classes"].eq(2), "Binario", "Multiclasse")
df_classes["pct_classe_minoritaria"] = (df_classes["classe_minoritaria_qtd"] / df_classes["numero_linhas"] * 100).clip(lower=0, upper=100)
df_classes["pct_classe_majoritaria"] = (df_classes["classe_majoritaria_qtd"] / df_classes["numero_linhas"] * 100).clip(lower=0, upper=100)

minoria_ok = df_classes["pct_classe_minoritaria"].ge(10)
df_selecionados = df_classes[minoria_ok].copy()
SELECIONADOS_CSV = ROOT / "catalogo_datasets_selecionados_minoria10.csv"
df_selecionados.to_csv(SELECIONADOS_CSV, index=False)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#2f3542",
    "axes.labelcolor": "#18212f",
    "axes.titleweight": "bold",
    "axes.titlesize": 20,
    "axes.titlepad": 16,
    "axes.labelsize": 16,
    "xtick.color": "#2f3542",
    "ytick.color": "#2f3542",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "font.size": 15,
    "legend.frameon": False,
})

COLORS = {
    "blue": "#2563EB",
    "green": "#059669",
    "orange": "#F59E0B",
    "red": "#DC2626",
    "purple": "#7C3AED",
    "cyan": "#0891B2",
    "gray": "#64748B",
    "light_gray": "#E2E8F0",
    "ink": "#0F172A",
}

def fmt_int(value, _pos=None):
    return f"{int(value):,}".replace(",", ".")

def fmt_pct(value, _pos=None):
    return f"{value:.0f}%"

def style_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.25)
    ax.spines["bottom"].set_linewidth(1.25)
    ax.grid(axis=grid_axis, color="#CBD5E1", linewidth=1.0, alpha=0.75)
    ax.set_axisbelow(True)

def label_bars(ax, bars, values, fontsize=15):
    ymax = max(values) if len(values) else 0
    offset = max(ymax * 0.015, 0.6)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, fmt_int(value),
                ha="center", va="bottom", fontsize=fontsize, fontweight="bold", color=COLORS["ink"])

def save_current_fig(filename):
    path = FIG_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Figura salva em: {path.relative_to(ROOT)}")

print(f"Catalogo: {CATALOGO.relative_to(ROOT)}")
print(f"Datasets no catalogo: {len(df)}")
print(f"Datasets binarios ou multiclasse usados nos graficos de classe: {len(df_classes)}")
print(f"Datasets com uma unica classe detectada e excluidos desses graficos: {(df['numero_classes'] == 1).sum()}")
print(f"Datasets selecionados com minoria >= 10%: {len(df_selecionados)}")
print(f"CSV de selecionados: {SELECIONADOS_CSV.relative_to(ROOT)}")

## 1. Distribuicao binario vs multiclasse

Calculada a partir de `numero_classes`: `2` classes = binario; `3` a `10` classes = multiclasse.

In [ ]:
counts = df_classes["tipo_dataset"].value_counts().reindex(["Binario", "Multiclasse"], fill_value=0)
total = int(counts.sum())

fig, ax = plt.subplots(figsize=(9.4, 6.4), constrained_layout=True)
wedges, _texts, autotexts = ax.pie(
    counts.values,
    labels=None,
    colors=[COLORS["blue"], COLORS["green"]],
    startangle=90,
    counterclock=False,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * total / 100))})",
    textprops={"color": COLORS["ink"], "fontsize": 17, "fontweight": "bold"},
    wedgeprops={"linewidth": 2.2, "edgecolor": "white"},
)
for item in autotexts:
    item.set_fontweight("bold")
    item.set_fontsize(16)
legend_labels = [f"{label}: {fmt_int(value)} datasets" for label, value in counts.items()]
ax.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(0.88, 0.5), fontsize=16)
ax.set_title(f"Distribuicao por tipo de dataset (n={total})")
ax.axis("equal")
save_current_fig("01_pizza_binario_multiclasse_por_numero_classes.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 2. Histograma do numero de linhas

Escala logaritmica no eixo x por causa da assimetria forte nos tamanhos dos datasets.

In [ ]:
values = df["numero_linhas"].dropna()
values = values[values > 0]
bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 16)

fig, ax = plt.subplots(figsize=(9.2, 5.8), constrained_layout=True)
ax.hist(values, bins=bins, color=COLORS["orange"], edgecolor="white", linewidth=1.3, alpha=0.92)
ax.set_xscale("log")
ax.set_title("Histograma do numero de linhas")
ax.set_xlabel("Numero de linhas (escala log)")
ax.set_ylabel("Quantidade de datasets")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_int))
style_axis(ax)
save_current_fig("02_histograma_numero_linhas.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 3. Histograma do numero de features

Tambem usa escala logaritmica no eixo x, porque poucos datasets tem muitas features.

In [ ]:
values = df["features"].dropna()
values = values[values > 0]
bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 16)

fig, ax = plt.subplots(figsize=(9.2, 5.8), constrained_layout=True)
ax.hist(values, bins=bins, color=COLORS["cyan"], edgecolor="white", linewidth=1.3, alpha=0.92)
ax.set_xscale("log")
ax.set_title("Histograma do numero de features")
ax.set_xlabel("Numero de features (escala log)")
ax.set_ylabel("Quantidade de datasets")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_int))
style_axis(ax)
save_current_fig("03_histograma_numero_features.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 4. Numero de datasets por numero de classes

Este grafico usa o mesmo universo do grafico de pizza, portanto a soma das barras deve ser igual ao `n` do primeiro plot.

In [1]:
counts = df_classes["numero_classes"].astype(int).value_counts().reindex(range(2, 11), fill_value=0)
assert int(counts.sum()) == len(df_classes)

fig, ax = plt.subplots(figsize=(9.2, 5.8), constrained_layout=True)
bars = ax.bar(counts.index.astype(str), counts.values, color=COLORS["green"], edgecolor="white", linewidth=1.2)
label_bars(ax, bars, counts.values)
ax.set_title(f"Datasets por numero de classes (n={int(counts.sum())})")
ax.set_xlabel("Numero de classes")
ax.set_ylabel("Quantidade de datasets")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_int))
style_axis(ax)
save_current_fig("04_barras_numero_classes.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

NameError: name 'df_classes' is not defined

## 5. Boxplot do percentual da classe minoritaria

Percentual calculado como `classe_minoritaria_qtd / numero_linhas * 100`, usando apenas datasets com 2 a 10 classes.

In [ ]:
values = df_classes["pct_classe_minoritaria"].dropna()
rng = np.random.default_rng(42)
x = np.ones(len(values)) + rng.normal(0, 0.025, len(values))

fig, ax = plt.subplots(figsize=(7.4, 5.8), constrained_layout=True)
ax.boxplot(values, vert=True, widths=0.34, patch_artist=True,
           boxprops={"facecolor": COLORS["light_gray"], "edgecolor": COLORS["gray"], "linewidth": 1.8},
           medianprops={"color": COLORS["red"], "linewidth": 3},
           whiskerprops={"color": COLORS["gray"], "linewidth": 1.8}, capprops={"color": COLORS["gray"], "linewidth": 1.8},
           flierprops={"marker": "o", "markersize": 0})
ax.scatter(x, values, s=42, alpha=0.42, color=COLORS["orange"], edgecolor="white", linewidth=0.4)
ax.set_ylim(0, 55)
ax.set_xticks([1])
ax.set_xticklabels(["Classe minoritaria"])
ax.set_title("Percentual da classe minoritaria por dataset")
ax.set_ylabel("Percentual da classe minoritaria")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_pct))
style_axis(ax)
save_current_fig("05_boxplot_percentual_classe_minoritaria.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 6. Boxplot do percentual da classe majoritaria

Percentual calculado como `classe_majoritaria_qtd / numero_linhas * 100`, usando apenas datasets com 2 a 10 classes.

In [ ]:
values = df_classes["pct_classe_majoritaria"].dropna()
rng = np.random.default_rng(43)
x = np.ones(len(values)) + rng.normal(0, 0.025, len(values))

fig, ax = plt.subplots(figsize=(7.4, 5.8), constrained_layout=True)
ax.boxplot(values, vert=True, widths=0.34, patch_artist=True,
           boxprops={"facecolor": COLORS["light_gray"], "edgecolor": COLORS["gray"], "linewidth": 1.8},
           medianprops={"color": COLORS["red"], "linewidth": 3},
           whiskerprops={"color": COLORS["gray"], "linewidth": 1.8}, capprops={"color": COLORS["gray"], "linewidth": 1.8},
           flierprops={"marker": "o", "markersize": 0})
ax.scatter(x, values, s=42, alpha=0.42, color=COLORS["purple"], edgecolor="white", linewidth=0.4)
ax.set_ylim(0, 105)
ax.set_xticks([1])
ax.set_xticklabels(["Classe majoritaria"])
ax.set_title("Percentual da classe majoritaria por dataset")
ax.set_ylabel("Percentual da classe majoritaria")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_pct))
style_axis(ax)
save_current_fig("06_boxplot_percentual_classe_majoritaria.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 7. Percentual de dados ausentes

Usa somente a coluna `percentual_ausentes`, convertida para porcentagem.

In [ ]:
percent = (df["percentual_ausentes"].clip(lower=0) * 100).fillna(0)
bins = [-1e-12, 0, 1, 5, 10, 20, np.inf]
labels = ["0%", "(0, 1%]", "(1, 5%]", "(5, 10%]", "(10, 20%]", ">20%"]
bucket = pd.cut(percent, bins=bins, labels=labels, include_lowest=True, right=True)
counts = bucket.value_counts().reindex(labels, fill_value=0)

fig, ax = plt.subplots(figsize=(10.0, 5.8), constrained_layout=True)
bar_colors = [COLORS["gray"], COLORS["blue"], COLORS["orange"], COLORS["red"], COLORS["purple"], COLORS["ink"]]
bars = ax.bar(counts.index.astype(str), counts.values, color=bar_colors, edgecolor="white", linewidth=1.2)
label_bars(ax, bars, counts.values)
ax.set_title("Datasets por percentual de dados ausentes")
ax.set_xlabel("Percentual de valores ausentes")
ax.set_ylabel("Quantidade de datasets")
ax.yaxis.set_major_formatter(FuncFormatter(fmt_int))
style_axis(ax)
save_current_fig("07_percentual_dados_ausentes.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 8. Distribuicao apos filtros de selecao

Aqui selecionamos apenas datasets com classe minoritaria >= 10%. Dados ausentes nao entram neste filtro. A pizza continua sendo calculada por `numero_classes`.

In [ ]:
counts = df_selecionados["tipo_dataset"].value_counts().reindex(["Binario", "Multiclasse"], fill_value=0)
total = int(counts.sum())
removidos_minoria = int((df_classes["pct_classe_minoritaria"] < 10).sum())
mantidos_com_ausentes = int((df_selecionados["percentual_ausentes"] > 0).sum())

fig, ax = plt.subplots(figsize=(9.4, 6.4), constrained_layout=True)
wedges, _texts, autotexts = ax.pie(
    counts.values,
    labels=None,
    colors=[COLORS["blue"], COLORS["green"]],
    startangle=90,
    counterclock=False,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * total / 100))})" if total else "",
    textprops={"color": COLORS["ink"], "fontsize": 17, "fontweight": "bold"},
    wedgeprops={"linewidth": 2.2, "edgecolor": "white"},
)
for item in autotexts:
    item.set_fontweight("bold")
    item.set_fontsize(16)
legend_labels = [f"{label}: {fmt_int(value)} datasets" for label, value in counts.items()]
ax.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(0.88, 0.5), fontsize=16)
ax.set_title(f"Distribuicao com minoria >= 10% (n={total})")
ax.axis("equal")
print(f"Removidos por classe minoritaria < 10% no universo 2-10 classes: {removidos_minoria}")
print(f"Datasets com ausentes mantidos na selecao: {mantidos_com_ausentes}")
print(f"Selecionados finais: {total}")
save_current_fig("08_pizza_selecionados_minoria10.png")
if "agg" not in plt.get_backend().lower():
    plt.show()

## 9. Pizza apos remover a classe minoritaria antiga

Esta analise combina datasets que ja tinham classe minoritaria >= 10% com os multiclasse recuperaveis, sem filtrar por dados ausentes. Para datasets com minoria original < 10%, os binarios sao descartados; nos multiclasse, remove em memoria as linhas da classe minoritaria antiga, recalcula a distribuicao e mantem apenas os que ficam com nova minoria >= 10%.

In [ ]:
def read_dataset_csv(path):
    attempts = (
        {"encoding": "utf-8", "low_memory": False},
        {"encoding": "latin1", "low_memory": False},
        {"encoding": "utf-8", "sep": None, "engine": "python"},
        {"encoding": "latin1", "sep": None, "engine": "python"},
    )
    last_error = None
    for kwargs in attempts:
        try:
            return pd.read_csv(path, na_values=("?", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none"), keep_default_na=True, **kwargs)
        except Exception as exc:
            last_error = exc
    raise ValueError(f"Nao foi possivel ler {path}: {last_error}")

base_minoria_baixa = df_classes[df_classes["pct_classe_minoritaria"].lt(10)].copy()

recalculados = []
for _, row in base_minoria_baixa.iterrows():
    numero_classes_original = int(row["numero_classes"])
    classe_removida = str(row["classe_minoritaria"])
    classe_removida_registro = ""
    linhas_removidas = 0

    if numero_classes_original == 2:
        new_counts = pd.Series({
            str(row["classe_majoritaria"]): int(row["classe_majoritaria_qtd"]),
            str(row["classe_minoritaria"]): int(row["classe_minoritaria_qtd"]),
        })
        linhas_restantes = int(row["numero_linhas"])
        numero_classes_recalculado = numero_classes_original
    else:
        data = read_dataset_csv(PROJECT_ROOT / row["caminho_relativo"])
        target_col = str(row["coluna_classe"])
        target_values = data[target_col].astype("string").fillna("<NA>")
        data_filtrado = data.loc[target_values.ne(classe_removida)].copy()
        new_counts = data_filtrado[target_col].astype("string").fillna("<NA>").value_counts(dropna=False)
        classe_removida_registro = classe_removida
        linhas_removidas = int(row["classe_minoritaria_qtd"])
        linhas_restantes = int(new_counts.sum()) if len(new_counts) else 0
        numero_classes_recalculado = int(len(new_counts))

    if numero_classes_original == 2:
        nova_minoria = str(new_counts.index[-1])
        nova_majoritaria = str(new_counts.index[0])
        nova_minoria_pct = float(new_counts.iloc[-1] / new_counts.sum() * 100)
        tipo_recalculado = "Binario"
        status = "descartado_binario_minoria_menor_10"
    elif len(new_counts) < 2:
        status = "descartado_apos_remocao_ficou_1_classe"
        nova_minoria_pct = np.nan
        tipo_recalculado = "Uma classe"
        nova_minoria = ""
        nova_majoritaria = str(new_counts.index[0]) if len(new_counts) else ""
    else:
        nova_minoria = str(new_counts.index[-1])
        nova_majoritaria = str(new_counts.index[0])
        nova_minoria_pct = float(new_counts.iloc[-1] / new_counts.sum() * 100)
        tipo_recalculado = "Binario" if len(new_counts) == 2 else "Multiclasse"
        status = "selecionado" if nova_minoria_pct >= 10 else "descartado_nova_minoria_menor_10"

    recalculados.append({
        "nome_arquivo": row["nome_arquivo"],
        "caminho_relativo": row["caminho_relativo"],
        "numero_classes_original": numero_classes_original,
        "classe_minoritaria_original": classe_removida,
        "classe_minoritaria_removida": classe_removida_registro,
        "pct_classe_minoritaria_original": row["pct_classe_minoritaria"],
        "linhas_originais": int(row["numero_linhas"]),
        "linhas_removidas": linhas_removidas,
        "linhas_restantes": linhas_restantes,
        "numero_classes_recalculado": numero_classes_recalculado,
        "tipo_dataset_recalculado": tipo_recalculado,
        "nova_classe_minoritaria": nova_minoria,
        "nova_classe_majoritaria": nova_majoritaria,
        "pct_nova_classe_minoritaria": nova_minoria_pct,
        "status": status,
    })

recalculados_df = pd.DataFrame(recalculados)
recalculados_csv = ROOT / "catalogo_datasets_minoria_removida_recalculado.csv"
recalculados_df.to_csv(recalculados_csv, index=False)

selecionados_recalculados = recalculados_df[recalculados_df["status"].eq("selecionado")].copy()
selecionados_originais = df_selecionados[["nome_arquivo", "caminho_relativo", "tipo_dataset"]].copy()
selecionados_originais["origem_selecao"] = "original_minoria_ok"

recuperados = selecionados_recalculados[["nome_arquivo", "caminho_relativo"]].copy()
recuperados["tipo_dataset"] = selecionados_recalculados["tipo_dataset_recalculado"].values
recuperados["origem_selecao"] = "recuperado_apos_remover_minoria"

selecionados_finais = pd.concat([selecionados_originais, recuperados], ignore_index=True)
paths_com_ausentes = set(df_classes.loc[df_classes["percentual_ausentes"].fillna(0).gt(0), "caminho_relativo"])
mantidos_com_ausentes_final = int(selecionados_finais["caminho_relativo"].isin(paths_com_ausentes).sum())
selecionados_finais_csv = ROOT / "catalogo_datasets_selecionados_com_minoria_removida_recalculado.csv"
selecionados_finais.to_csv(selecionados_finais_csv, index=False)

counts = selecionados_finais["tipo_dataset"].value_counts().reindex(["Binario", "Multiclasse"], fill_value=0)
total = int(counts.sum())
pie_counts = counts[counts > 0]

fig, ax = plt.subplots(figsize=(9.4, 6.4), constrained_layout=True)
wedges, _texts, autotexts = ax.pie(
    pie_counts.values,
    labels=None,
    colors=[COLORS["blue"] if label == "Binario" else COLORS["green"] for label in pie_counts.index],
    startangle=90,
    counterclock=False,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * total / 100))})" if pct > 0 and total else "",
    textprops={"color": COLORS["ink"], "fontsize": 17, "fontweight": "bold"},
    wedgeprops={"linewidth": 2.2, "edgecolor": "white"},
)
for item in autotexts:
    item.set_fontweight("bold")
    item.set_fontsize(16)
legend_labels = [f"{label}: {fmt_int(value)} datasets" for label, value in counts.items() if value > 0]
ax.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(0.88, 0.5), fontsize=16)
ax.set_title(f"Selecionados finais com recuperados (n={total})")
ax.axis("equal")

print(f"Datasets com minoria original < 10%: {len(base_minoria_baixa)}")
print(recalculados_df["status"].value_counts().to_string())
print(f"Recuperados apos remover minoria antiga: {len(selecionados_recalculados)}")
print(f"Datasets com ausentes mantidos na selecao final: {mantidos_com_ausentes_final}")
print(f"Selecionados finais para a pizza: {total}")
print(f"CSV recalculado: {recalculados_csv.relative_to(ROOT)}")
print(f"CSV selecionados finais: {selecionados_finais_csv.relative_to(ROOT)}")
save_current_fig("09_pizza_minoria_removida_recalculada_minoria10.png")
if "agg" not in plt.get_backend().lower():
    plt.show()